# Fake.br: experimento One-Class SVM para desvio linguístico

Notebook independente derivado do protocolo do Isolation Forest. O detector aprende somente
com notícias **True**; labels Fake aparecem apenas na seleção assistida por validação e na
avaliação externa. `anomalyScore` não é probabilidade de falsidade e um alerta não comprova
que a notícia seja falsa.

**Recorte implementado:** matriz fechada de quatro valores de `nu`, seleção ROC-AUC → AP →
menor `nu` na validação, decisões nativa e q95 separadas, avaliação única no teste e controle
Isolation Forest no mesmo split. Visualizações completas e casos extremos ficam fora deste recorte.


## 1. Imports e configuração do ambiente

Execute esta célula em um kernel limpo. No Colab, as bibliotecas `numpy`, `pandas`, `scikit-learn` e `matplotlib` normalmente já estão disponíveis; se o ambiente solicitar, instale-as antes de executar todas as células.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
from itertools import combinations
from importlib.metadata import version
import hashlib
import platform
import json
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay,
)

In [ ]:
RANDOM_STATE = 42
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
print("Python:", platform.python_version())
print({name: version(name) for name in ["numpy", "pandas", "scikit-learn", "matplotlib", "ipykernel"]})
BASE_NOTEBOOK_SHA256 = "25ff9d35c233c9bd6067ce0ebcf14bc78a077659f5c769a15b8f4486bd542d79"
EXPERIMENT_KEY = "one-class-svm-core-confirmatory"


## 3. Carregamento dos dados

Mesmo ZIP e revisão fixa do notebook original. Textos completos e metadados são
carregados para auditoria; o truncamento acontece ANTES da extração usada nos
modelos. Nenhuma notícia é excluída por comprimento.

In [ ]:
CORPUS_REVISION = "780f5516c4ae070761632d98ac3368f3ded09d35"
CORPUS_URL = f"https://codeload.github.com/roneysco/Fake.br-Corpus/zip/{CORPUS_REVISION}"
projectFolder = Path.cwd()
dataFolder = projectFolder / "data"
dataFolder.mkdir(exist_ok=True)
archivePath = dataFolder / f"Fake.br-Corpus-{CORPUS_REVISION}.zip"

if not archivePath.exists():
    temporaryPath = archivePath.with_suffix(".download")
    with urlopen(CORPUS_URL, timeout=120) as response, temporaryPath.open("wb") as target:
        while chunk := response.read(1024 * 1024):
            target.write(chunk)
    with ZipFile(temporaryPath) as archive:
        assert archive.testzip() is None, "ZIP corrompido; refaça o download."
    temporaryPath.replace(archivePath)

print("Corpus revision:", CORPUS_REVISION)
print("Archive SHA256:", hashlib.sha256(archivePath.read_bytes()).hexdigest())

In [ ]:
metadataColumns = [
    "autor", "link", "categoria", "data_publicacao",
    "num_tokens", "num_palavras", "num_types", "num_links", "num_maiusculas",
    "num_verbos", "num_verbos_subj_imp", "num_substantivos", "num_adjetivos",
    "num_adverbios", "num_verbos_modais", "num_pron_1_2_sing", "num_pron_1_plural",
    "num_pronomes", "pausalidade", "num_caracteres", "tam_medio_sentenca",
    "tam_medio_palavra", "pct_erros_ortograficos", "emotividade", "diversidade",
]

In [ ]:
def loadNewsTexts(archive, folder, label):
    records = []
    for name in sorted(archive.namelist()):
        if f"/full_texts/{folder}/" not in name or not name.endswith(".txt"):
            continue
        articleId = Path(name).stem + ("t" if label == 0 else "")
        records.append({
            "id": articleId, "text": archive.read(name).decode("utf-8"),
            "label": label, "sourceClass": folder,
        })
    assert records, f"Nenhum texto encontrado em {folder}"
    return pd.DataFrame(records)


def loadNewsMetadata(archive, folder, label):
    records = []
    for name in sorted(archive.namelist()):
        if f"/full_texts/{folder}-meta-information/" not in name or not name.endswith("-meta.txt"):
            continue
        values = [line.strip() for line in archive.read(name).decode("utf-8").splitlines()]
        assert len(values) == len(metadataColumns), f"Esquema inesperado em {name}: {len(values)} linhas"
        articleId = Path(name).name.removesuffix("-meta.txt") + ("t" if label == 0 else "")
        records.append({**dict(zip(metadataColumns, values)), "id": articleId, "metadataLabel": label})
    assert records, f"Nenhum metadado encontrado em {folder}"
    return pd.DataFrame(records)

In [ ]:
with ZipFile(archivePath) as archive:
    textsFrame = pd.concat([
        loadNewsTexts(archive, "fake", 1), loadNewsTexts(archive, "true", 0),
    ], ignore_index=True)
    metadataFrame = pd.concat([
        loadNewsMetadata(archive, "fake", 1), loadNewsMetadata(archive, "true", 0),
    ], ignore_index=True)

assert textsFrame["id"].is_unique and metadataFrame["id"].is_unique
assert set(textsFrame["id"]) == set(metadataFrame["id"]), "Texto/metadados sem correspondência"
newsFrame = textsFrame.merge(metadataFrame, on="id", validate="one_to_one", indicator=True)
assert newsFrame["_merge"].eq("both").all()
assert newsFrame["label"].eq(newsFrame["metadataLabel"]).all()
newsFrame = newsFrame.drop(columns=["_merge", "metadataLabel"])
assert newsFrame["text"].str.strip().ne("").all()
print(f"{len(newsFrame):,} notícias carregadas; textos e metadados correspondem 1:1.")

## 4. Contrato dos labels

**0 = True; 1 = Fake.** O rótulo forma as partições e permite avaliação externa.
Não entra como feature. A origem nas pastas também é verificada.

In [ ]:
labelNames = {0: "True", 1: "Fake"}
assert labelNames == {0: "True", 1: "Fake"}
assert set(newsFrame["label"].unique()) == {0, 1}
assert newsFrame.loc[newsFrame["label"].eq(0), "sourceClass"].eq("true").all()
assert newsFrame.loc[newsFrame["label"].eq(1), "sourceClass"].eq("fake").all()
display(newsFrame.groupby(["label", "sourceClass"]).size().rename("newsCount").to_frame())

## 5. Preparação dos metadados

Preservamos todos os campos brutos em `rawNewsFrame` e todas as contagens numéricas
em `newsFrame`. Valores não numéricos viram NaN com contagem explícita; nenhum
ausente é preenchido antes do treino. `tem_autor` segue a lógica do original:
ausência para string vazia, `None`, `none` ou `NULL`. Não é uma medida de credibilidade.

In [ ]:
rawNewsFrame = newsFrame.copy(deep=True)
numericMetadataColumns = metadataColumns[4:]
convertedMetadata = newsFrame[numericMetadataColumns].apply(pd.to_numeric, errors="coerce")
conversionMissing = convertedMetadata.isna().sum().rename("missingAfterNumericConversion")
display(conversionMissing.to_frame())
newsFrame[numericMetadataColumns] = convertedMetadata
newsFrame["tem_autor"] = (~newsFrame["autor"].fillna("").astype(str).str.strip().isin(
    ["", "None", "none", "NULL"]
)).astype(int)

## 6. Extração no texto efetivamente utilizado

Normalização Unicode NFKC, remoção do BOM inicial e primeiros CHARACTER_LIMIT
caracteres, incluindo espaços/pontuação. Textos menores permanecem menores, sem
preenchimento. O corte pode dividir palavras/frases. Palavras são sequências de
letras com hífen/apóstrofo interno; tokens também incluem números e pontuação.
Tipos são palavras distintas ignorando caixa. TTR = tipos/tokens; diversidade =
tipos/palavras. Maiúsculas conta palavras totalmente maiúsculas com mais de uma
letra. Links são URLs http(s) presentes no corpo. Estas definições explícitas
não pretendem reproduzir o extrator desconhecido dos metadados históricos.

As demais contagens linguísticas são marcadas ausentes na cópia de features,
não imputadas nem selecionadas pelo modelo; originais ficam em rawNewsFrame e
newsFrame. Autor permanece como metadado válido da notícia inteira.

In [ ]:
def calculateRatio(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce").astype(float)
    denominator = pd.to_numeric(denominator, errors="coerce").astype(float)
    safeDenominator = denominator.where(denominator.gt(0) & np.isfinite(denominator))
    return numerator.div(safeDenominator).replace([np.inf, -np.inf], np.nan)

In [ ]:
import re
import unicodedata

CHARACTER_LIMIT = 300
numericMetadataColumns = metadataColumns[4:]
anomalyColumns = ["tem_autor", "typeTokenRatio", "linkDensity", "punctuationDensity", "uppercaseRatio", "diversidade"]
wordPattern = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*", re.UNICODE)
tokenPattern = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*|\d+(?:[.,]\d+)*|[^\w\s]", re.UNICODE)


def measureText(text, characterLimit):
    normalized = unicodedata.normalize("NFKC", text).lstrip("\ufeff")
    if characterLimit is not None:
        if not isinstance(characterLimit, int) or characterLimit < 1:
            raise ValueError("Limite deve ser inteiro positivo ou None.")
        normalized = normalized[:characterLimit]
    words = wordPattern.findall(normalized)
    return {"text": normalized, "num_palavras": len(words),
            "num_tokens": len(tokenPattern.findall(normalized)),
            "num_types": len({word.casefold() for word in words}),
            "num_links": len(re.findall(r"https?://\S+", normalized, flags=re.IGNORECASE)),
            "num_maiusculas": sum(word.isupper() and len(word) > 1 for word in words),
            "num_caracteres": len(normalized)}


def extractAnomalyFeatures(newsFrame, characterLimit=CHARACTER_LIMIT):
    featuresFrame = newsFrame.copy(deep=True)
    featuresFrame["num_palavras_original"] = newsFrame["num_palavras"]
    featuresFrame["text_original"] = newsFrame["text"]
    featuresFrame[numericMetadataColumns] = np.nan
    measurements = pd.DataFrame([measureText(text, characterLimit) for text in newsFrame["text"]], index=newsFrame.index)
    for column in measurements:
        featuresFrame[column] = measurements[column]
    featuresFrame["typeTokenRatio"] = calculateRatio(featuresFrame["num_types"], featuresFrame["num_tokens"])
    featuresFrame["diversidade"] = calculateRatio(featuresFrame["num_types"], featuresFrame["num_palavras"])
    featuresFrame["linkDensity"] = calculateRatio(featuresFrame["num_links"], featuresFrame["num_palavras"])
    featuresFrame["punctuationDensity"] = calculateRatio(featuresFrame["num_tokens"] - featuresFrame["num_palavras"], featuresFrame["num_tokens"])
    featuresFrame["uppercaseRatio"] = calculateRatio(featuresFrame["num_maiusculas"], featuresFrame["num_palavras"])
    return featuresFrame

In [ ]:
featuresFrame = extractAnomalyFeatures(newsFrame)
featuresFrame[anomalyColumns] = featuresFrame[anomalyColumns].replace([np.inf, -np.inf], np.nan)
display(featuresFrame[anomalyColumns].isna().sum().rename("missingCount").to_frame())
print("Limite de caracteres:", CHARACTER_LIMIT)
display(featuresFrame.groupby("label")[["num_palavras_original", "num_palavras", "num_caracteres"]].agg(["min", "median", "max"]))
print("Textos menores que o limite:", int(featuresFrame["num_caracteres"].lt(CHARACTER_LIMIT).sum()))

## 7. Sanity checks

As seis features utilizam o prefixo; a presença de autor vem dos metadados.
Contagens originais ficam preservadas para auditoria e não alimentam o modelo.

In [ ]:
assert "num_palavras" not in anomalyColumns
assert "label" not in anomalyColumns and "id" not in anomalyColumns
assert len(anomalyColumns) == len(set(anomalyColumns)) == 6
assert featuresFrame["id"].is_unique
assert featuresFrame["num_caracteres"].le(CHARACTER_LIMIT).all()
assert featuresFrame["num_palavras_original"].equals(newsFrame["num_palavras"])
assert featuresFrame["num_verbos"].isna().all()
assert not np.isinf(featuresFrame[anomalyColumns].to_numpy(dtype=float)).any()
assert measureText("casa " * 100, 300)["num_palavras"] == 60
assert measureText("casa " * 100, 300)["num_caracteres"] == 300
print("Contagens do prefixo verificadas; num_palavras não entra no treinamento.")

## 8. Análise estatística

Estatísticas descritivas por label, sem substituir vetores individuais por médias.
Esta inspeção do corpus completo atende ao protocolo exploratório: não usamos
seus resultados para selecionar features, ajustar hiperparâmetros ou threshold.
Qualquer escolha futura guiada por estes resultados exige nova avaliação independente.
Correlações de Pearson com comprimento são mostradas no total e por classe para
evitar confundir efeitos de classe e de tamanho. NaN em correlação pode indicar
feature constante. Mantemos as seis features recalculadas, inclusive possíveis redundâncias.

In [ ]:
statisticsRecords = []
for label, group in featuresFrame.groupby("label", sort=True):
    for feature in anomalyColumns:
        values = group[feature]
        q1, q3 = values.quantile([0.25, 0.75])
        statisticsRecords.append({
            "label": label, "class": labelNames[label], "feature": feature,
            "count": values.count(), "mean": values.mean(), "median": values.median(),
            "std": values.std(), "min": values.min(), "Q1": q1, "Q3": q3,
            "IQR": q3 - q1, "max": values.max(), "missingCount": values.isna().sum(),
        })
featureStatistics = pd.DataFrame(statisticsRecords).set_index(["label", "class", "feature"])
display(featureStatistics)

lengthColumns = ["num_palavras", "num_tokens"]
lengthCorrelations = pd.concat({
    name: group[anomalyColumns + lengthColumns].corr().loc[anomalyColumns, lengthColumns]
    for name, group in [
        ("All", featuresFrame),
        ("True (0)", featuresFrame.loc[featuresFrame["label"].eq(0)]),
        ("Fake (1)", featuresFrame.loc[featuresFrame["label"].eq(1)]),
    ]
}, names=["group", "feature"])
display(lengthCorrelations)
featureCorrelations = featuresFrame[anomalyColumns].corr()
display(featureCorrelations.round(3))
display(lengthCorrelations.xs("typeTokenRatio", level="feature"))
print("Pearson TTR vs diversidade:", featureCorrelations.loc["typeTokenRatio", "diversidade"])

## 9. Train / Validation / Test

True: 60% treino, 20% validação, 20% teste. Fake: 50% validação, 50% teste.
Todas as divisões usam `random_state=42`. Verificamos IDs exclusivos e cobertura
integral do corpus. **Nenhuma notícia Fake participa de fit ou calibração.**

Limitação do protocolo solicitado: a divisão é por notícia, não por assunto,
fonte ou data. O corpus possui pares True/Fake com o mesmo número-base; IDs
`123t` e `123` são notícias distintas, mas podem tratar do mesmo assunto em
partições diferentes. IDs exclusivos não demonstram independência temática.

In [ ]:
normalFrame = featuresFrame.loc[featuresFrame["label"].eq(0)].copy()
fakeFrame = featuresFrame.loc[featuresFrame["label"].eq(1)].copy()
normalTrainFrame, normalHoldoutFrame = train_test_split(
    normalFrame, train_size=0.60, random_state=RANDOM_STATE,
)
normalValidationFrame, normalTestFrame = train_test_split(
    normalHoldoutFrame, test_size=0.50, random_state=RANDOM_STATE,
)
fakeValidationFrame, fakeTestFrame = train_test_split(
    fakeFrame, test_size=0.50, random_state=RANDOM_STATE,
)
partitions = {
    "normalTrain": normalTrainFrame, "normalValidation": normalValidationFrame,
    "normalTest": normalTestFrame, "fakeValidation": fakeValidationFrame,
    "fakeTest": fakeTestFrame,
}
for name, frame in partitions.items():
    assert not frame.empty and frame["id"].is_unique
    assert frame["label"].eq(0 if name.startswith("normal") else 1).all()
for (leftName, left), (rightName, right) in combinations(partitions.items(), 2):
    assert set(left["id"]).isdisjoint(right["id"]), f"IDs compartilhados: {leftName}/{rightName}"
assert set().union(*(set(frame["id"]) for frame in partitions.values())) == set(featuresFrame["id"])
assert sum(len(frame) for frame in partitions.values()) == len(featuresFrame)
assert normalTrainFrame["label"].eq(0).all()
display(pd.DataFrame([
    {"partition": name, "count": len(frame), "label": int(frame["label"].iloc[0])}
    for name, frame in partitions.items()
]).set_index("partition"))

## 10. One-Class SVM e guardas True-only

Cada candidato possui imputer, scaler e detector próprios. O `.fit()` recebe somente
`normalTrainFrame`. `nu` é um limite teórico superior para erros de treino e inferior para
vetores de suporte, sujeito ao ajuste e a efeitos numéricos; as frações observadas são medidas.


In [ ]:
ONE_CLASS_CANDIDATES = {
    "ocsvm_nu_001": 0.01,
    "ocsvm_nu_0025": 0.025,
    "ocsvm_nu_005": 0.05,
    "ocsvm_nu_010": 0.10,
}


def buildOneClassPipeline(nu):
    if nu not in ONE_CLASS_CANDIDATES.values():
        raise ValueError(f"nu fora da matriz pré-registrada: {nu}")
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("detector", OneClassSVM(kernel="rbf", gamma="scale", nu=nu)),
    ])


def fitNormalOnly(pipeline, trainingFrame):
    if trainingFrame.empty or not trainingFrame["label"].eq(0).all():
        raise ValueError("Treino permitido apenas com notícias True (label == 0).")
    trainingFeatures = trainingFrame[anomalyColumns]
    if np.isinf(trainingFeatures.to_numpy(dtype=float)).any():
        raise ValueError("Features de treino contêm infinito.")
    emptyColumns = trainingFeatures.columns[trainingFeatures.isna().all()].tolist()
    if emptyColumns:
        raise ValueError(f"Features inteiramente ausentes em normalTrain: {emptyColumns}")
    return pipeline.fit(trainingFeatures)


def anomalyScores(pipeline, frame):
    scores = -pipeline.decision_function(frame[anomalyColumns])
    if not np.isfinite(scores).all():
        raise ValueError("Scores não finitos.")
    return scores


def nativeFlags(pipeline, frame):
    decision_flags = pipeline.decision_function(frame[anomalyColumns]) < 0
    prediction_flags = pipeline.predict(frame[anomalyColumns]) == -1
    if not np.array_equal(decision_flags, prediction_flags):
        raise AssertionError("Fronteira nativa divergiu de predict == -1.")
    return decision_flags


def binaryMetrics(labels, scores, flags):
    labels = np.asarray(labels, dtype=int)
    flags = np.asarray(flags, dtype=bool)
    matrix = confusion_matrix(labels, flags.astype(int), labels=[0, 1])
    tn, fp, fn, tp = (int(value) for value in matrix.ravel())
    return {
        "rocAuc": float(roc_auc_score(labels, scores)),
        "averagePrecision": float(average_precision_score(labels, scores)),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "accuracy": float((tp + tn) / matrix.sum()),
        "precision": float(precision_score(labels, flags, zero_division=0)),
        "recall": float(recall_score(labels, flags, zero_division=0)),
        "f1": float(f1_score(labels, flags, zero_division=0)),
        "fpr": float(fp / (tn + fp)) if tn + fp else 0.0,
    }


def selectCandidate(validationRecords, tolerance=1e-12):
    if not validationRecords:
        raise ValueError("A seleção exige resultados de validação.")
    best_auc = max(record["validationRocAuc"] for record in validationRecords)
    auc_ties = [record for record in validationRecords if best_auc - record["validationRocAuc"] <= tolerance]
    best_ap = max(record["validationAveragePrecision"] for record in auc_ties)
    ap_ties = [record for record in auc_ties if best_ap - record["validationAveragePrecision"] <= tolerance]
    return min(ap_ties, key=lambda record: record["nu"])["modelKey"]


## 11. Treino, validação e seleção congelada

Os quatro pipelines são ajustados em 2.160 True. O q95 usa exclusivamente as 720 True de
validação. Fake-validation assiste somente a escolha de `nu` por ROC-AUC, AP e menor `nu`.
Nenhuma métrica de teste existe quando `selectedModelKey` é materializado.


In [ ]:
validationFrame = pd.concat([normalValidationFrame, fakeValidationFrame], ignore_index=True)
validationLabels = validationFrame["label"].to_numpy(dtype=int)
candidatePipelines = {}
validationRecords = []

for modelKey, nu in ONE_CLASS_CANDIDATES.items():
    pipeline = buildOneClassPipeline(nu)
    fitStarted = time.perf_counter()
    fitNormalOnly(pipeline, normalTrainFrame)
    fitSeconds = time.perf_counter() - fitStarted
    scoreStarted = time.perf_counter()
    normalValidationScores = anomalyScores(pipeline, normalValidationFrame)
    validationScores = anomalyScores(pipeline, validationFrame)
    validationNativeFlags = nativeFlags(pipeline, validationFrame)
    scoreSeconds = time.perf_counter() - scoreStarted
    q95Threshold = float(np.quantile(normalValidationScores, 0.95))
    validationQ95Flags = validationScores >= q95Threshold
    native = binaryMetrics(validationLabels, validationScores, validationNativeFlags)
    q95 = binaryMetrics(validationLabels, validationScores, validationQ95Flags)
    trainNativeFlags = nativeFlags(pipeline, normalTrainFrame)
    detector = pipeline["detector"]
    record = {
        "modelKey": modelKey, "nu": nu, "kernel": detector.kernel,
        "resolvedGamma": float(detector._gamma),
        "normalTrainCount": len(normalTrainFrame),
        "normalValidationCount": len(normalValidationFrame),
        "fakeValidationCount": len(fakeValidationFrame),
        "imputerMedian": pipeline["imputer"].statistics_.astype(float).tolist(),
        "scalerMean": pipeline["scaler"].mean_.astype(float).tolist(),
        "scalerScale": pipeline["scaler"].scale_.astype(float).tolist(),
        "supportVectorCount": int(detector.support_.size),
        "supportVectorFraction": float(detector.support_.size / len(normalTrainFrame)),
        "normalTrainNativeOutlierCount": int(trainNativeFlags.sum()),
        "normalTrainNativeOutlierFraction": float(trainNativeFlags.mean()),
        "fitSeconds": fitSeconds, "validationScoreSeconds": scoreSeconds,
        "q95Threshold": q95Threshold,
        "normalValidationQ95AlertRate": float(np.mean(normalValidationScores >= q95Threshold)),
        "validationRocAuc": q95["rocAuc"],
        "validationAveragePrecision": q95["averagePrecision"],
        "nativeMetrics": native, "q95Metrics": q95,
    }
    np.testing.assert_allclose(
        pipeline["imputer"].statistics_, normalTrainFrame[anomalyColumns].median().to_numpy()
    )
    imputedTrain = pipeline["imputer"].transform(normalTrainFrame[anomalyColumns])
    np.testing.assert_allclose(pipeline["scaler"].mean_, imputedTrain.mean(axis=0))
    expectedScale = imputedTrain.std(axis=0)
    expectedScale[expectedScale == 0] = 1.0
    np.testing.assert_allclose(pipeline["scaler"].scale_, expectedScale)
    candidatePipelines[modelKey] = pipeline
    validationRecords.append(record)

selectedModelKey = selectCandidate(validationRecords)
selectedPipeline = candidatePipelines[selectedModelKey]
selectedValidationRecord = next(record for record in validationRecords if record["modelKey"] == selectedModelKey)
selectedQ95Threshold = selectedValidationRecord["q95Threshold"]
validationResultsFrame = pd.DataFrame(validationRecords)
display(validationResultsFrame[[
    "modelKey", "nu", "resolvedGamma", "supportVectorCount", "supportVectorFraction",
    "normalTrainNativeOutlierFraction", "q95Threshold", "normalValidationQ95AlertRate",
    "validationRocAuc", "validationAveragePrecision",
]])
print("Candidato congelado antes do teste:", selectedModelKey)


## 12. Controle Isolation Forest e avaliação final única

O controle usa o mesmo treino, features, IDs e q95 True-validation. Somente o candidato OCSVM
selecionado recebe métricas de teste. A fronteira nativa do OCSVM permanece separada do q95.


In [ ]:
controlPipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("detector", IsolationForest(
        n_estimators=300, contamination="auto", random_state=RANDOM_STATE, n_jobs=-1,
    )),
])
fitNormalOnly(controlPipeline, normalTrainFrame)
controlNormalValidationScores = anomalyScores(controlPipeline, normalValidationFrame)
controlQ95Threshold = float(np.quantile(controlNormalValidationScores, 0.95))

testFrame = pd.concat([normalTestFrame, fakeTestFrame], ignore_index=True)
testLabels = testFrame["label"].to_numpy(dtype=int)
selectedScores = anomalyScores(selectedPipeline, testFrame)
selectedNativeFlags = nativeFlags(selectedPipeline, testFrame)
selectedQ95Flags = selectedScores >= selectedQ95Threshold
controlScores = anomalyScores(controlPipeline, testFrame)
controlQ95Flags = controlScores >= controlQ95Threshold

selectedNativeMetrics = binaryMetrics(testLabels, selectedScores, selectedNativeFlags)
selectedQ95Metrics = binaryMetrics(testLabels, selectedScores, selectedQ95Flags)
controlQ95Metrics = binaryMetrics(testLabels, controlScores, controlQ95Flags)
scoreDistributions = pd.concat({
    "ocsvm": pd.DataFrame({"label": testLabels, "score": selectedScores}).groupby("label")["score"].agg(["count", "mean", "median", "std", "min", "max"]),
    "isolationForest": pd.DataFrame({"label": testLabels, "score": controlScores}).groupby("label")["score"].agg(["count", "mean", "median", "std", "min", "max"]),
}, names=["model", "label"])

testPredictionsFrame = testFrame[["id", "label"] + anomalyColumns].copy()
testPredictionsFrame["ocsvmAnomalyScore"] = selectedScores
testPredictionsFrame["ocsvmNativeIsAnomaly"] = selectedNativeFlags
testPredictionsFrame["ocsvmQ95IsAnomaly"] = selectedQ95Flags
testPredictionsFrame["isolationForestAnomalyScore"] = controlScores
testPredictionsFrame["isolationForestQ95IsAnomaly"] = controlQ95Flags
assert testPredictionsFrame["id"].is_unique

transitionLabels = np.select(
    [selectedQ95Flags & controlQ95Flags, ~selectedQ95Flags & ~controlQ95Flags, selectedQ95Flags & ~controlQ95Flags],
    ["alerta_mantido", "sem_alerta", "alerta_adicionado_ocsvm"],
    default="alerta_removido_ocsvm",
)
testPredictionsFrame["transition"] = transitionLabels
transitionCounts = testPredictionsFrame.groupby(["label", "transition"]).size().rename("count").reset_index()
assert int(transitionCounts["count"].sum()) == len(testPredictionsFrame)

testResults = {
    "selectedModelKey": selectedModelKey,
    "fakePrevalenceApBaseline": float(np.mean(testLabels == 1)),
    "ocsvmQ95Threshold": selectedQ95Threshold,
    "isolationForestQ95Threshold": controlQ95Threshold,
    "ocsvmNative": selectedNativeMetrics,
    "ocsvmQ95": selectedQ95Metrics,
    "isolationForestQ95": controlQ95Metrics,
    "transitions": transitionCounts.to_dict(orient="records"),
}
display(scoreDistributions)
display(pd.DataFrame({
    "OCSVM nativo": selectedNativeMetrics,
    "OCSVM q95": selectedQ95Metrics,
    "Isolation Forest q95": controlQ95Metrics,
}).T)
display(transitionCounts)


## 13. Exportação opcional e conclusão calculada

Quando `ONE_CLASS_SVM_OUTPUT_DIR` existe, o `Run All` salva resultados estruturados nessa
pasta nova. Sem a variável, o notebook permanece interativo e não grava artefatos experimentais.


In [ ]:
def jsonReady(value):
    if isinstance(value, dict):
        return {str(key): jsonReady(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonReady(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    return value


manifest = {
    "experimentKey": EXPERIMENT_KEY,
    "baseNotebookSha256": BASE_NOTEBOOK_SHA256,
    "corpusRevision": CORPUS_REVISION,
    "seed": RANDOM_STATE,
    "characterLimit": CHARACTER_LIMIT,
    "features": anomalyColumns,
    "partitionCounts": {name: len(frame) for name, frame in partitions.items()},
    "candidates": ONE_CLASS_CANDIDATES,
    "selectedModelKey": selectedModelKey,
    "selectionUsesFakeValidationLabels": True,
    "testWasNotUsedForSelection": True,
}
outputDirectory = os.environ.get("ONE_CLASS_SVM_OUTPUT_DIR")
if outputDirectory:
    outputPath = Path(outputDirectory)
    outputPath.mkdir(parents=True, exist_ok=False) if not outputPath.exists() else None
    (outputPath / "manifest.json").write_text(json.dumps(jsonReady(manifest), indent=2, allow_nan=False), encoding="utf-8")
    (outputPath / "validation-results.json").write_text(json.dumps(jsonReady(validationRecords), indent=2, allow_nan=False), encoding="utf-8")
    (outputPath / "test-results.json").write_text(json.dumps(jsonReady(testResults), indent=2, allow_nan=False), encoding="utf-8")
    testPredictionsFrame.to_csv(outputPath / "test-predictions.csv", index=False)
    transitionCounts.to_csv(outputPath / "transitions.csv", index=False)

display(Markdown(f'''O candidato congelado foi **{selectedModelKey}**. No teste histórico, OCSVM q95 obteve
ROC-AUC **{selectedQ95Metrics['rocAuc']:.4f}**, AP **{selectedQ95Metrics['averagePrecision']:.4f}**,
recall **{selectedQ95Metrics['recall']:.2%}** e FPR **{selectedQ95Metrics['fpr']:.2%}**.
O controle Isolation Forest q95 obteve ROC-AUC **{controlQ95Metrics['rocAuc']:.4f}**,
AP **{controlQ95Metrics['averagePrecision']:.4f}**, recall **{controlQ95Metrics['recall']:.2%}**
e FPR **{controlQ95Metrics['fpr']:.2%}**.

Estes números descrevem desvio linguístico neste split histórico; não provam falsidade nem
superioridade estatística. A seleção de `nu` foi assistida por labels Fake de validação. O texto
foi truncado em 300 caracteres, o split não é temático e o teste histórico já foi consultado.
**Colab não foi validado neste recorte.**
'''))
